# Run Searches and Filters on Existing Tables

## Introduction

In the previous tutorial, we saw how to extend `BaseTableSchema` and `TableManifestation` to add custom search methods. However, sometimes you may want to run a one-off query or perform complex filtering without modifying your class definitions.

This tutorial demonstrates how to run **runtime searches and filters** using a `TableManifestation`'s built-in session management and its reference to the underlying SQLAlchemy model.

### Learning Objectives

- Running custom SQLAlchemy queries using `TableManifestation.create_session()`.
- Accessing the SQLAlchemy model via `TableManifestation.table_schema`.
- Implementing runtime filters for both **synchronous** and **asynchronous** workflows.

**Prerequisites:**
- Completion of the [BaseTable Tutorial](basetable_tutorial.ipynb).
- Basic knowledge of SQLAlchemy's `select` and `where` syntax.


## Importing the Module

We import the necessary components from **sqlalchemyobjects** and SQLAlchemy.


In [ ]:
from pathlib import Path
from typing import Any
from sqlalchemy import select, and_, or_
from sqlalchemy.orm import Mapped, mapped_column, DeclarativeBase
from sqlalchemy.ext.asyncio import AsyncAttrs

from sqlalchemyobjects import Database, BaseTableSchema, TableManifestation


## 1. Define a Standard Table

For this tutorial, we use a standard table without any custom methods.


In [ ]:
class UserTableSchema(BaseTableSchema):
    """A standard user table schema without custom query logic."""
    name: Mapped[str] = mapped_column()
    role: Mapped[str] = mapped_column(default="user")
    age: Mapped[int] = mapped_column()

class DatabaseSchema(AsyncAttrs, DeclarativeBase):
    """Declarative base for the database."""

class UserTable(UserTableSchema, DatabaseSchema):
    """The concrete User table."""
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)

class MyDatabase(Database):
    """Database class managing the User table."""
    schema = DatabaseSchema
    table_map = {
        "users": (TableManifestation, UserTable, {})
    }

    @property
    def users(self) -> TableManifestation:
        return self.tables["users"]


## 2. Initialization and Data Insertion

Initialize the database and insert some sample data.


In [ ]:
db_path = Path("runtime_search_tutorial.sqlite")
if db_path.exists():
    db_path.unlink()

database = MyDatabase(path=db_path)
database.create_database()

# Insert sample data using built-in CRUD
database.users.insert_all([
    {"name": "Alice", "role": "admin", "age": 30},
    {"name": "Bob", "role": "user", "age": 25},
    {"name": "Charlie", "role": "admin", "age": 45},
    {"name": "David", "role": "user", "age": 20},
    {"name": "Eve", "role": "user", "age": 35},
])


## 3. Runtime Searches (Synchronous)

To run a custom search at runtime, we use `create_session()` from the manifestation and `select()` from SQLAlchemy. We can access the table class via `manifestation.table_schema`.


In [ ]:
# Example: Find all admins over 30
users_manifestation = database.users
User = users_manifestation.table_schema

with users_manifestation.create_session() as session:
    # Build the statement using the table_schema
    stmt = select(User).where(and_(User.role == "admin", User.age > 30))

    # Execute and get results
    results = session.execute(stmt).scalars().all()

    print("Admins over 30:")
    for user in results:
        print(f" - {user.name} ({user.age})")


## 4. Runtime Searches (Asynchronous)

The same principle applies to asynchronous operations. Use `create_async_session()` and `stream()` (or `execute()`).


In [ ]:
import anyio

async def async_runtime_demo():
    async_db_path = anyio.Path("runtime_search_async.sqlite")
    if await async_db_path.exists():
        await async_db_path.unlink()

    # Initialize with async_engine=True
    async_db = MyDatabase(path=str(async_db_path), async_engine=True)
    await async_db.create_database_async()

    # Insert sample data
    await async_db.users.insert_all_async([
        {"name": "Alice", "role": "admin", "age": 30},
        {"name": "Bob", "role": "user", "age": 25},
    ])

    # Runtime Async Search
    users_async = async_db.users
    User = users_async.table_schema

    async with users_async.create_async_session() as session:
        stmt = select(User).where(User.age < 30)
        result = await session.stream(stmt)

        print("\n[Async] Users under 30:")
        async for user in result.scalars():
            print(f" - {user.name} ({user.age})")

    await async_db.close_async()
    await async_db_path.unlink()

# Run the async function
await async_runtime_demo()


## Why use Runtime Searches?

1. **Flexibility**: You can build any query on the fly without needing to anticipate every search requirement in your class definitions.
2. **Reduced Boilerplate**: For simple or infrequent queries, creating a custom `TableManifestation` subclass might be overkill.
3. **Dynamic Filters**: Runtime searches are ideal for applications where filters are generated dynamically (e.g., from a web request's query parameters).

## Best Practices

- **Access via `table_schema`**: Always use `manifestation.table_schema` to refer to the SQLAlchemy model to ensure your code is decoupled from the specific class name.
- **Context Managers**: Always use `with` or `async with` when calling `create_session()` or `create_async_session()` to ensure sessions are properly closed.
- **Limit Results**: When performing runtime searches on large tables, always use `.limit()` and `.offset()` to manage performance.


## Conclusion

While extending tables is great for common and reusable searches, performing runtime searches provides the flexibility needed for one-off queries and dynamic filtering. By leveraging `create_session()` and `table_schema`, you can fully utilize SQLAlchemy's power while still benefiting from **sqlalchemyobjects**' session management.
